# ENSO × A股种植业：状态依赖性检验  
# ENSO × A-Share Planting Sector: Regime-Dependent Factor Test

**研究目标 / Research objective**

检验 Daily Niño 3.4 SST Anomaly 是否与申万种植业指数（801016）的未来收益有关，并进一步验证这种关系是否具有**状态依赖性（regime dependence）**。

Test whether the Daily Niño 3.4 SST Anomaly is related to future returns of the Shenwan Planting Industry Index (801016), and whether the ENSO beta changes across market regimes.

**保留的核心分析 / Core analyses retained**
1. 构造 Daily Niño 3.4 的 1 日滞后变量，降低前视偏差。  
   Use a one-day lag of Daily Niño 3.4 to reduce look-ahead bias.
2. 检验最近 5 年与最近 15 年的 1D / 5D / 20D / 60D / 90D 关系。  
   Compare recent 5-year and 15-year relationships across multiple horizons.
3. 对三个外部事件节点后的样本分别进行 5D HAC 回归。  
   Run 5D HAC regressions within three externally defined event windows.
4. 以 2026-07-03 为状态切换节点，用交互项正式检验 ENSO Beta 是否改变。  
   Formally test the beta shift after 2026-07-03 using an interaction term.

> 注：事件节点用于划分市场状态，不代表这些事件被证明“导致”了 Beta 变化。  
> Note: Event dates define regimes; the regressions do not establish causal effects of the events themselves.


In [1]:
# ============================================================
# 0. 导入库与显示设置
# 0. Imports and display settings
# ============================================================

from pathlib import Path

import akshare as ak
import pandas as pd
import statsmodels.api as sm

pd.options.display.float_format = "{:.6f}".format


## 1. 数据准备 / Data Preparation

In [2]:
# ============================================================
# 1.1 读取 Daily Niño 3.4 SST Anomaly
# 1.1 Load Daily Niño 3.4 SST Anomaly
#
# 文件应至少包含：
# Date, Nino34_Daily
#
# The CSV should contain at least:
# Date, Nino34_Daily
# ============================================================

# Support running Jupyter from either the repository root or notebooks/.
# 支持从仓库根目录或 notebooks/ 目录启动 Jupyter。
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "nino34_daily.csv"
results_dir = project_root / "results" / "tables"
results_dir.mkdir(parents=True, exist_ok=True)

nino34 = pd.read_csv(
    data_path,
    parse_dates=["Date"]
)

nino34 = (
    nino34
    .sort_values("Date")
    .reset_index(drop=True)
)

# 将气候变量滞后1个自然日，降低前视偏差
# Lag the climate variable by one calendar day to reduce look-ahead bias
nino34["Nino34_Lag1D"] = nino34["Nino34_Daily"].shift(1)

print(nino34[["Date", "Nino34_Daily", "Nino34_Lag1D"]].head())
print(nino34[["Date", "Nino34_Daily", "Nino34_Lag1D"]].tail())


        Date  Nino34_Daily  Nino34_Lag1D
0 2000-01-01     -1.878836           NaN
1 2000-01-02     -1.953510     -1.878836
2 2000-01-03     -1.966761     -1.953510
3 2000-01-04     -1.930476     -1.966761
4 2000-01-05     -1.873611     -1.930476
           Date  Nino34_Daily  Nino34_Lag1D
9728 2026-08-20      2.615540      2.625770
9729 2026-08-21      2.644204      2.615540
9730 2026-08-22      2.699826      2.644204
9731 2026-08-23      2.689605      2.699826
9732 2026-08-24      2.690090      2.689605


In [3]:
# ============================================================
# 1.2 获取申万种植业指数 801016
# 1.2 Download Shenwan Planting Industry Index 801016
# ============================================================

planting = ak.index_hist_sw(
    symbol="801016",
    period="day"
)

planting = planting[["日期", "收盘"]].copy()
planting["日期"] = pd.to_datetime(planting["日期"])

planting = (
    planting
    .sort_values("日期")
    .reset_index(drop=True)
)

print(planting.head())
print(planting.tail())


          日期          收盘
0 1999-12-30 1000.000000
1 2000-01-04 1028.540000
2 2000-01-05 1015.880000
3 2000-01-06 1058.160000
4 2000-01-07 1106.420000
             日期          收盘
6445 2026-09-01 2803.250000
6446 2026-09-02 2636.990000
6447 2026-09-03 2647.040000
6448 2026-09-04 2731.730000
6449 2026-09-07 2854.020000


In [4]:
# ============================================================
# 1.3 合并种植业指数与 Daily Niño 3.4
# 1.3 Merge planting-sector data with Daily Niño 3.4
#
# Daily Niño 3.4 是日频自然日数据，因此按交易日期精确匹配即可。
# Daily Niño 3.4 is available on calendar days, so exact-date
# matching is sufficient for A-share trading dates.
# ============================================================

planting_nino = pd.merge(
    planting,
    nino34[["Date", "Nino34_Daily", "Nino34_Lag1D"]],
    left_on="日期",
    right_on="Date",
    how="left"
).drop(columns=["Date"])

planting_nino = (
    planting_nino
    .sort_values("日期")
    .reset_index(drop=True)
)

# ENSO 数据仍可用的最后一个交易日
# Last trading day with an available lagged ENSO observation
analysis_end_date = planting_nino.loc[
    planting_nino["Nino34_Lag1D"].notna(),
    "日期"
].max()

print("Analysis end date / 研究截止日:", analysis_end_date)
print("Missing values / 缺失值：")
print(planting_nino[["收盘", "Nino34_Lag1D"]].isna().sum())


Analysis end date / 研究截止日: 2026-08-24 00:00:00
Missing values / 缺失值：
收盘               0
Nino34_Lag1D    11
dtype: int64


## 2. 通用函数 / Reusable Functions

下方只保留两组通用函数，避免在不同时间窗口中重复写相同的回归代码。  
The following reusable functions replace repeated regression code across multiple sample windows.


In [5]:
# ============================================================
# 2.1 构造 Forward Returns
# 2.1 Construct forward returns
#
# Fwd_hD(t) = P(t+h) / P(t) - 1
# ============================================================

def add_forward_returns(df, horizons=(1, 5, 20, 60, 90)):
    data = df.copy()

    for h in horizons:
        data[f"Fwd_{h}D"] = (
            data["收盘"].shift(-h)
            / data["收盘"]
            - 1
        )

    return data


# ============================================================
# 2.2 HAC / Newey-West 回归
# 2.2 HAC / Newey-West regression
#
# Fwd_Return(t,h)
# = alpha + beta * Nino34(t-1) + epsilon(t)
#
# 对 h 日重叠收益率，初始设定 maxlags = h - 1。
# For overlapping h-day returns, use maxlags = h - 1
# as the baseline HAC specification.
# ============================================================

def run_hac_regression(df, y_col, x_col="Nino34_Lag1D", maxlags=0):
    reg_data = df[[y_col, x_col]].dropna()

    y = reg_data[y_col]
    X = sm.add_constant(reg_data[[x_col]])

    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags}
    )

    return model

## 3. 历史窗口稳定性：最近 5 年 vs 最近 15 年  
## Historical Stability: Recent 5 Years vs Recent 15 Years

这一部分用于判断 ENSO 与种植业收益的关系是否长期稳定。  
This section checks whether the ENSO-return relationship is stable across a recent and a longer historical window.

> 已修正原 notebook 中“注释写 15 年、代码实际使用 `years=10`”的不一致；这里真正使用 15 年。  
> The original notebook labelled this test as 15 years but used `years=10`. This cleaned version correctly uses 15 years.


In [6]:
# ============================================================
# 3.1 一次性运行多个 horizon
# 3.1 Run multiple return horizons in one function
# ============================================================

def run_horizon_panel(df, years, end_date):
    start_date = end_date - pd.DateOffset(years=years)

    sample = df[
        (df["日期"] >= start_date) &
        (df["日期"] <= end_date)
    ].copy()

    horizons = [1, 5, 20, 60, 90]

    # Compute targets only after defining the sample window. This ensures
    # every forward return ends on or before the stated analysis end date.
    # 先截取样本窗口，再计算未来收益，避免使用研究截止日后的价格。
    sample = add_forward_returns(sample, horizons=horizons)
    rows = []

    for h in horizons:
        model = run_hac_regression(
            sample,
            y_col=f"Fwd_{h}D",
            x_col="Nino34_Lag1D",
            maxlags=max(h - 1, 0)
        )

        rows.append({
            "Sample_Window": f"{years}Y",
            "Horizon": f"{h}D",
            "Beta": model.params["Nino34_Lag1D"],
            "HAC_SE": model.bse["Nino34_Lag1D"],
            "z_stat": model.tvalues["Nino34_Lag1D"],
            "p_value": model.pvalues["Nino34_Lag1D"],
            "R_squared": model.rsquared,
            "N": int(model.nobs)
        })

    return pd.DataFrame(rows)


results_5y = run_horizon_panel(
    planting_nino,
    years=5,
    end_date=analysis_end_date
)

results_15y = run_horizon_panel(
    planting_nino,
    years=15,
    end_date=analysis_end_date
)

historical_results = pd.concat(
    [results_5y, results_15y],
    ignore_index=True
)

print(historical_results)


  Sample_Window Horizon      Beta   HAC_SE    z_stat  p_value  R_squared     N
0            5Y      1D -0.000617 0.000575 -1.074558 0.282573   0.001048  1201
1            5Y      5D -0.002818 0.002223 -1.267755 0.204886   0.004573  1197
2            5Y     20D -0.013539 0.007496 -1.806078 0.070906   0.032994  1182
3            5Y     60D -0.038947 0.010603 -3.673251 0.000239   0.098315  1142
4            5Y     90D -0.040908 0.014358 -2.849174 0.004383   0.084862  1112
5           15Y      1D -0.000193 0.000448 -0.430785 0.666625   0.000068  3624
6           15Y      5D -0.000866 0.001855 -0.466700 0.640714   0.000274  3620
7           15Y     20D -0.003827 0.006296 -0.607819 0.543307   0.001471  3605
8           15Y     60D -0.009103 0.014427 -0.630942 0.528079   0.002772  3565
9           15Y     90D  0.000057 0.022338  0.002543 0.997971   0.000000  3535


### 结果阅读 / How to read this table

重点观察 / Focus on:
- `Beta`：ENSO 与未来收益的方向和经济量级。  
  Direction and economic magnitude of ENSO exposure.
- `p_value`：HAC 修正后的显著性。  
  HAC-adjusted statistical significance.
- 5Y 与 15Y 是否表现出明显不同。  
  Whether the 5Y and 15Y windows show different relationships.

如果近期窗口显著、长期窗口不稳定，这只是**状态依赖性的线索**，不能仅凭不同窗口的 p 值直接证明 regime shift。  
If the recent window is significant but the long window is not, that motivates a regime hypothesis but does not by itself prove a structural beta shift.


## 4. 事件窗口回归 / Event-Window Regressions

只使用事件发生后的数据，并统一以 `Fwd_5D` 为因变量。  
Each regression uses only observations after the corresponding event and the same 5-day forward return as the dependent variable.

**事件节点 / Event dates**
- 2026-02-28：美伊冲突节点 / US-Iran conflict event
- 2026-07-03：WMO El Niño 信息节点 / WMO El Niño information event
- 2026-07-31：WMO 强 El Niño 信息节点 / WMO strong El Niño information event

关键修正：**先截取事件窗口，再在窗口内部计算 `Fwd_5D`**，因此不会使用研究截止日之后的股票价格。  
Key correction: `Fwd_5D` is calculated *after* slicing each event window, preventing prices after the analysis end date from leaking into the regression.


In [7]:
# ============================================================
# 4.1 事件窗口回归
# 4.1 Event-window regressions
# ============================================================

event_dates = {
    "US_Iran_War": pd.Timestamp("2026-02-28"),
    "WMO_ElNino_Confirmed": pd.Timestamp("2026-07-03"),
    "WMO_Strong_ElNino": pd.Timestamp("2026-07-31")
}


def significance_label(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return "Not Significant"


event_results = []
event_models = {}

for event_name, event_date in event_dates.items():

    # 找到事件后的第一个A股交易日
    # Find the first A-share trading day on or after the event
    available_dates = planting_nino.loc[
        (planting_nino["日期"] >= event_date) &
        (planting_nino["日期"] <= analysis_end_date),
        "日期"
    ]

    if available_dates.empty:
        continue

    first_trading_date = available_dates.iloc[0]

    # 只截取事件发生后至研究截止日的数据
    # Keep only the event-to-end-date window
    event_sample = planting_nino[
        (planting_nino["日期"] >= first_trading_date) &
        (planting_nino["日期"] <= analysis_end_date)
    ].copy()

    # 在事件窗口内部重新构造 5D forward return
    # Recompute 5D forward returns inside the event window
    # so the final 5 rows naturally become NaN
    event_sample["Fwd_5D"] = (
        event_sample["收盘"].shift(-5)
        / event_sample["收盘"]
        - 1
    )

    trading_days = event_sample["日期"].nunique()

    model = run_hac_regression(
        event_sample,
        y_col="Fwd_5D",
        x_col="Nino34_Lag1D",
        maxlags=4
    )

    event_models[event_name] = model

    event_results.append({
        "Event": event_name,
        "Event_Date": event_date,
        "First_Trading_Date": first_trading_date,
        "End_Date": analysis_end_date,
        "Trading_Days": trading_days,
        "Valid_Observations": int(model.nobs),
        "Return_Horizon": "5D",
        "Beta": model.params["Nino34_Lag1D"],
        "HAC_SE": model.bse["Nino34_Lag1D"],
        "z_stat": model.tvalues["Nino34_Lag1D"],
        "p_value": model.pvalues["Nino34_Lag1D"],
        "R_squared": model.rsquared,
        "N": int(model.nobs)
    })


results_event_window = pd.DataFrame(event_results)

results_event_window["Significance"] = (
    results_event_window["p_value"]
    .apply(significance_label)
)

print(results_event_window)


                  Event Event_Date First_Trading_Date   End_Date  \
0           US_Iran_War 2026-02-28         2026-03-02 2026-08-24   
1  WMO_ElNino_Confirmed 2026-07-03         2026-07-03 2026-08-24   
2     WMO_Strong_ElNino 2026-07-31         2026-07-31 2026-08-24   

   Trading_Days  Valid_Observations Return_Horizon     Beta   HAC_SE   z_stat  \
0           121                 116             5D 0.013423 0.008675 1.547331   
1            37                  32             5D 0.081309 0.038029 2.138063   
2            17                  12             5D 0.277920 0.201693 1.377936   

   p_value  R_squared    N     Significance  
0 0.121783   0.071460  116  Not Significant  
1 0.032512   0.231466   32               **  
2 0.168223   0.095089   12  Not Significant  


## 5. 正式检验 Beta 是否改变 / Formal Beta-Shift Test

以 **2026-07-03** 为外生状态切换节点，使用最近 5 年样本估计：

\[
Fwd5D_t
=
\alpha
+
\beta_1 Nino34_{t-1}
+
\beta_2 Post_t
+
\beta_3(Nino34_{t-1} \times Post_t)
+
\varepsilon_t
\]

其中 / where:
- `β1`：事件前 ENSO Beta / pre-event ENSO beta
- `β2`：事件后的平均收益水平变化 / intercept-level shift after the event
- `β3`：**事件后 ENSO Beta 相对事件前的变化 / change in ENSO beta after the event**
- `β_post = β1 + β3`

核心假设 / Main hypothesis:

\[
H_0:\beta_3=0
\quad\text{vs}\quad
H_1:\beta_3\neq0
\]

若 `ENSO_Post` 的 p-value 小于预先设定的显著性水平，则有统计证据表明 ENSO Beta 在该状态节点前后发生变化。  
If the p-value of `ENSO_Post` is below the pre-specified significance level, there is statistical evidence of a beta shift across the regime breakpoint.


In [8]:
# ============================================================
# 5.1 构建最近5年的 regime-shift 样本
# 5.1 Build the recent 5-year regime-shift sample
# ============================================================

regime_date = pd.Timestamp("2026-07-03")
analysis_start_date = analysis_end_date - pd.DateOffset(years=5)

regime_data = planting_nino[
    (planting_nino["日期"] >= analysis_start_date) &
    (planting_nino["日期"] <= analysis_end_date)
].copy()

# 在研究窗口内部重新计算 5D forward return
# Recompute the target inside the analysis window
regime_data["Fwd_5D"] = (
    regime_data["收盘"].shift(-5)
    / regime_data["收盘"]
    - 1
)

# Post = 0：事件前；Post = 1：事件后
# Post = 0 before the event; Post = 1 on/after the event
regime_data["Post"] = (
    regime_data["日期"] >= regime_date
).astype(int)

# 交互项：用于正式检验 ENSO Beta 是否改变
# Interaction term: formally tests whether the ENSO beta changes
regime_data["ENSO_Post"] = (
    regime_data["Nino34_Lag1D"]
    * regime_data["Post"]
)

reg_data = regime_data[
    ["Fwd_5D", "Nino34_Lag1D", "Post", "ENSO_Post"]
].dropna()

print("Total observations / 总样本数:", len(reg_data))
print("Pre-event observations / 事件前样本:", (reg_data["Post"] == 0).sum())
print("Post-event observations / 事件后样本:", (reg_data["Post"] == 1).sum())


Total observations / 总样本数: 1197
Pre-event observations / 事件前样本: 1165
Post-event observations / 事件后样本: 32


In [9]:
# ============================================================
# 5.2 交互项回归：OLS + HAC / Newey-West
# 5.2 Interaction regression: OLS + HAC / Newey-West
# ============================================================

y = reg_data["Fwd_5D"]

X = sm.add_constant(
    reg_data[
        ["Nino34_Lag1D", "Post", "ENSO_Post"]
    ]
)

regime_model = sm.OLS(
    y,
    X
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(regime_model.summary())


                            OLS Regression Results                            
Dep. Variable:                 Fwd_5D   R-squared:                       0.030
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     4.887
Date:                Wed, 09 Sep 2026   Prob (F-statistic):            0.00222
Time:                        10:30:27   Log-Likelihood:                 2175.2
No. Observations:                1197   AIC:                            -4342.
Df Residuals:                    1193   BIC:                            -4322.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0003      0.002     -0.137   

In [10]:
# ============================================================
# 5.3 提取并解释关键系数
# 5.3 Extract and interpret the key coefficients
# ============================================================

beta_pre = regime_model.params["Nino34_Lag1D"]
beta_change = regime_model.params["ENSO_Post"]
beta_post = beta_pre + beta_change

p_beta_pre = regime_model.pvalues["Nino34_Lag1D"]
p_beta_change = regime_model.pvalues["ENSO_Post"]

regime_summary = pd.DataFrame({
    "Metric": [
        "Pre-event ENSO Beta",
        "Beta Change After Event",
        "Post-event ENSO Beta",
        "Pre-event Beta p-value",
        "Beta Change p-value"
    ],
    "Value": [
        beta_pre,
        beta_change,
        beta_post,
        p_beta_pre,
        p_beta_change
    ]
})

print(regime_summary)

print("\n========== Conclusion / 结论 ==========")

if p_beta_change < 0.01:
    print(
        "ENSO Beta changed significantly at the 1% level. / "
        "ENSO Beta 在1%显著性水平下发生显著变化。"
    )
elif p_beta_change < 0.05:
    print(
        "ENSO Beta changed significantly at the 5% level. / "
        "ENSO Beta 在5%显著性水平下发生显著变化。"
    )
elif p_beta_change < 0.10:
    print(
        "ENSO Beta changed significantly at the 10% level. / "
        "ENSO Beta 在10%显著性水平下发生显著变化。"
    )
else:
    print(
        "No statistically significant beta shift is detected. / "
        "未发现 ENSO Beta 存在统计显著的结构变化。"
    )


                    Metric     Value
0      Pre-event ENSO Beta -0.005414
1  Beta Change After Event  0.086723
2     Post-event ENSO Beta  0.081309
3   Pre-event Beta p-value  0.013070
4      Beta Change p-value  0.022803

========== Conclusion / 结论 ==========
ENSO Beta changed significantly at the 5% level. / ENSO Beta 在5%显著性水平下发生显著变化。


## 6. 最终研究边界 / Final Research Boundary

本 notebook 将最终结论限制在**申万种植业指数（801016）**：

- 长期与近期结果并不完全一致，因此不把 ENSO 视为具有固定长期 Beta 的静态因子。  
  Long-run and recent results differ, so ENSO is not treated as a time-invariant factor.
- 事件窗口用于观察近期市场定价结构，但非常短的窗口（尤其最后一个事件窗口）只能作为探索性证据。  
  Event windows reveal recent pricing patterns, but very short windows are exploratory only.
- 交互项 `ENSO_Post` 是检验“Beta 是否发生结构变化”的核心统计量。  
  The `ENSO_Post` interaction coefficient is the formal test of a beta shift.
- 若 `ENSO_Post` 显著，可表述为：**ENSO 在 A 股种植业中具有状态依赖型定价效应的证据**。  
  If `ENSO_Post` is significant, the result supports evidence of a **regime-dependent ENSO pricing effect in the A-share planting sector**.
- 该结果是相关性和结构变化证据，并不证明 2026-07-03 的事件本身具有因果作用。  
  The result supports association and a structural beta change, not causal attribution to the event itself.

### 已删除的重复/非核心代码 / Removed redundant or non-core code
- 重复的 ONI 下载、清洗和多次 `merge_asof` 调试代码  
- ONI dummy vs Daily Niño 3.4 dummy 的中间比较
- 3Y / 1Y / 6M 多次重复窗口代码
- 养殖业的探索性回归
- 早期“事件决定 forward-return horizon”的错误研究设计
- 会使用窗口截止日之后股价的旧版 event-window 代码
- 申万农林牧渔一级指数的扩展检验（最终结论已限定在种植业）


In [11]:
# ============================================================
# 5.3 汇总并导出可复现结果
# 5.3 Summarise and export reproducible results
# ============================================================

beta_pre = regime_model.params["Nino34_Lag1D"]
beta_change = regime_model.params["ENSO_Post"]
beta_post = beta_pre + beta_change

regime_results = pd.DataFrame([{
    "Regime_Date": regime_date,
    "Pre_Event_Beta": beta_pre,
    "Pre_Event_p_value": regime_model.pvalues["Nino34_Lag1D"],
    "Beta_Change": beta_change,
    "Beta_Change_p_value": regime_model.pvalues["ENSO_Post"],
    "Post_Event_Beta": beta_post,
    "Pre_Event_N": int((reg_data["Post"] == 0).sum()),
    "Post_Event_N": int((reg_data["Post"] == 1).sum())
}])

historical_results.to_csv(
    results_dir / "historical_horizon_results.csv", index=False
)
results_event_window.to_csv(
    results_dir / "event_window_results.csv", index=False
)
regime_results.to_csv(
    results_dir / "regime_shift_results.csv", index=False
)

print(regime_results.to_string(index=False))

if regime_results.loc[0, "Beta_Change_p_value"] < 0.05:
    print(
        "\nEvidence of a beta change at the 5% level. / "
        "在5%显著性水平下，有证据表明 Beta 发生变化。"
    )
else:
    print(
        "\nNo evidence of a beta change at the 5% level. / "
        "在5%显著性水平下，没有足够证据表明 Beta 发生变化。"
    )

Regime_Date  Pre_Event_Beta  Pre_Event_p_value  Beta_Change  Beta_Change_p_value  Post_Event_Beta  Pre_Event_N  Post_Event_N
 2026-07-03       -0.005414           0.013070     0.086723             0.022803         0.081309         1165            32

Evidence of a beta change at the 5% level. / 在5%显著性水平下，有证据表明 Beta 发生变化。
